# 🐝 Lab 4.1 — Flow Matching From Scratch (Noise → Art in 2D)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab10_flow_matching_2d.ipynb)

⏱ **Time:** ~2 hours  |  📅 **Day 4 — Generative Worlds**

🎯 **Learning objectives** — by the end of this lab you will be able to:
- Place VAEs, DDPM diffusion, score matching and flow matching on one conceptual map, and say what each one learns.
- Derive and implement the **conditional flow matching** (CFM) loss with straight-line (rectified-flow) probability paths.
- Build an MLP **velocity field** with a sinusoidal time embedding and sample from it with **Euler** and **Heun (RK2)** ODE solvers.
- Measure how sample quality depends on the number of solver steps, and explain why **1 step collapses to the mean**.
- Run **reflow** (2-rectified flow) to straighten trajectories so that few-step sampling works.
- Train a **DDPM** ε-prediction model on the same data, sample it with ancestral and DDIM samplers, and convert between ε, score and velocity parameterizations.

🧰 **Prerequisites:** PyTorch basics (modules, optimizers, autograd), Gaussian distributions, a little calculus (derivatives, ODEs as "follow the arrow"). No GPU needed — everything here is 2D and trains in seconds.

**Why this matters in 2026.** The image, video and audio generators you've seen lately are, under the hood, mostly *learned vector fields that push noise into data*. Stable Diffusion 3 and FLUX.1 are trained with a rectified-flow objective, and many recent video models are diffusion or flow-matching transformers. The math is identical whether the "data" is a 2D point or a 4K video latent — so we'll learn it where we can *see every particle move*. Buzz the Backprop Bee 🐝 will be our target distribution. Let's turn noise into honey.

### 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| Setup | Imports, seeds, the hive color palette | 5 |
| 1 | Build 2D target distributions, including Buzz the bee and your own smiley | 15 |
| 2 | Map the family tree: VAE → DDPM → score matching → flow matching | 15 |
| 3 | Derive and implement the conditional flow matching loss; train six velocity fields | 20 |
| 4 | Write Euler and Heun samplers, watch particles fly (animated!), and measure steps vs quality | 25 |
| 5 | Reflow: straighten the paths and sample in 1–2 steps | 20 |
| 6 | Train DDPM on the same data, sample with DDIM, and translate between ε, score and velocity | 15 |
| Wrap-up | Boss Level, key takeaways, interview corner | 5 |

In [ ]:
# 🛠️ Setup — torch, numpy, sklearn, scipy and matplotlib are all preinstalled on Colab
import os, math, time, random, copy
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (tiny runs). Normal users: leave it off.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(42)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA", "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

print(f"torch {torch.__version__} | device = {DEVICE} | FAST = {FAST}")

## Part 1 — A zoo of 2D target distributions 🎨

A generative model needs a target $p_{\text{data}}$ that we can **sample** from but whose density we never write down. In 2D we can cheat and *look* at it, which makes every failure mode obvious.

We'll use classic benchmarks plus a couple of fun ones built from **parametric curves**:

| Target | What makes it hard |
|---|---|
| Two moons | two disconnected, curved modes |
| Spiral | long, thin, wrapped-around support |
| Checkerboard | 8 disconnected squares with sharp edges |
| Heart | a 1D curve living in 2D (zero area!) |
| 🐝 Buzz | a mixture of filled stripes, outlines and thin antennae |

All samplers return a `torch.float32` tensor of shape `(n, 2)`, scaled to roughly $[-2.5, 2.5]^2$ so they're comparable to a standard Gaussian.

In [ ]:
from sklearn.datasets import make_moons

def _t(x):
    return torch.as_tensor(np.asarray(x), dtype=torch.float32)

def sample_moons(n):
    x, _ = make_moons(n, noise=0.05)
    return _t((x - np.array([0.5, 0.25])) * 1.6)

def sample_spiral(n):
    t = np.sqrt(np.random.rand(n)) * 3 * np.pi           # sqrt -> roughly uniform density along the arc
    r = t / (3 * np.pi) * 2.3
    sign = np.where(np.random.rand(n) < 0.5, 1.0, -1.0)  # two interleaved arms
    x = np.stack([r * np.cos(t), r * np.sin(t)], 1) * sign[:, None]
    return _t(x + 0.05 * np.random.randn(n, 2))

def sample_checkerboard(n):
    x1 = np.random.rand(n) * 4 - 2
    x2 = np.random.rand(n) - np.random.randint(0, 2, n) * 2
    x2 = x2 + (np.floor(x1) % 2)
    return _t(np.stack([x1, x2], 1))

def sample_heart(n):
    t = np.random.rand(n) * 2 * np.pi
    x = 16 * np.sin(t) ** 3
    y = 13 * np.cos(t) - 5 * np.cos(2 * t) - 2 * np.cos(3 * t) - np.cos(4 * t)
    return _t(np.stack([x, y + 2.5], 1) / 7.5 + 0.03 * np.random.randn(n, 2))

def _ellipse_outline(n, cx, cy, a, b, rot=0.0):
    t = np.random.rand(n) * 2 * np.pi
    x, y = a * np.cos(t), b * np.sin(t)
    c, s = np.cos(rot), np.sin(rot)
    return np.stack([cx + c * x - s * y, cy + s * x + c * y], 1)

def _striped_body(n, a=1.3, b=0.8, band=0.36):
    out = np.zeros((0, 2))
    while len(out) < n:                                  # rejection sampling inside the ellipse
        p = np.random.rand(4 * n + 10, 2) * [2 * a, 2 * b] - [a, b]
        inside = (p[:, 0] / a) ** 2 + (p[:, 1] / b) ** 2 <= 1
        stripe = (np.floor((p[:, 0] + a) / band) % 2) == 1   # keep every other band -> stripes!
        out = np.concatenate([out, p[inside & stripe]])
    return out[:n]

def sample_bee(n):
    """Buzz the Backprop Bee 🐝 — a mixture of filled stripes, outlines and curves."""
    w = np.array([0.30, 0.16, 0.10, 0.26, 0.10, 0.08])   # stripes, body outline, head, wings, antennae, stinger
    k = np.random.multinomial(n, w / w.sum())
    stripes = _striped_body(k[0])
    outline = _ellipse_outline(k[1], 0, 0, 1.3, 0.8)
    head = _ellipse_outline(k[2], 1.68, 0.12, 0.38, 0.38)
    nw = k[3] // 2
    wings = np.concatenate([_ellipse_outline(nw, -0.3, 1.25, 0.38, 0.6, rot=0.3),
                            _ellipse_outline(k[3] - nw, 0.4, 1.2, 0.38, 0.6, rot=-0.3)])
    s = np.random.rand(k[4]); side = np.random.rand(k[4]) < 0.5      # two antennae: straight lines + a little bend
    base = np.where(side[:, None], [1.62, 0.48], [1.88, 0.42])
    tip = np.where(side[:, None], [1.55, 1.15], [2.25, 0.95])
    ant = base + s[:, None] * (tip - base) + np.stack([np.zeros_like(s), 0.15 * np.sin(np.pi * s)], 1)
    u = np.random.rand(k[5]); h = (np.random.rand(k[5]) - 0.5) * 0.3 * (1 - u)   # a small triangular stinger
    sting = np.stack([-1.25 - 0.45 * u, h], 1)
    pts = np.concatenate([stripes, outline, head, wings, ant, sting])
    pts = (pts - np.array([0.3, 0.5])) * 1.1 + 0.02 * np.random.randn(len(pts), 2)
    return _t(pts[np.random.permutation(len(pts))])

TARGETS = {"moons": sample_moons, "spiral": sample_spiral, "checkerboard": sample_checkerboard,
           "heart": sample_heart, "bee": sample_bee}

def scatter(ax, x, title=None, s=1.0, c=HIVE[1], lim=3.0, alpha=0.6):
    x = x.detach().cpu().numpy() if torch.is_tensor(x) else x
    ax.scatter(x[:, 0], x[:, 1], s=s, color=c, alpha=alpha, linewidths=0)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    if title: ax.set_title(title)

fig, axes = plt.subplots(1, len(TARGETS) + 1, figsize=(3 * (len(TARGETS) + 1), 3.2))
scatter(axes[0], torch.randn(5000, 2), "N(0, I)  (noise)", c=HIVE[4])
for ax, (name, fn) in zip(axes[1:], TARGETS.items()):
    scatter(ax, fn(5000), name)
plt.suptitle("Where we start (left) and where we want to go", y=1.02); plt.tight_layout(); plt.show()

### 🧪 Your Turn #1 — draw a smiley 🙂

Build `sample_smiley(n)` from parametric pieces:
- a **face**: circle outline, radius 2.0, centered at the origin
- two **eyes**: small *filled* discs of radius ~0.25 at $(\pm 0.7, 0.6)$
- a **mouth**: an arc of radius 1.2 centered at $(0, 0)$ spanning angles $200°\to 340°$

**Hints:** split `n` between the parts with `np.random.multinomial`; for a filled disc sample radius as $r = R\sqrt{U}$ (why the square root? area grows like $r^2$!); add a little Gaussian jitter (std ≈ 0.03) and return `_t(points)`.

In [ ]:
def sample_smiley(n):
    k = np.random.multinomial(n, [0.5, 0.15, 0.15, 0.2])   # face, left eye, right eye, mouth
    # TODO: face outline (radius 2.0)
    face = None
    # TODO: two filled eyes at (+-0.7, 0.6), radius 0.25   (r = R * sqrt(U))
    eye_l = None
    eye_r = None
    # TODO: mouth arc, radius 1.2, angles 200deg -> 340deg
    mouth = None
    # TODO: concatenate, add jitter, return _t(...)
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def _disc(n, cx, cy, R):
    r = R * np.sqrt(np.random.rand(n)); a = np.random.rand(n) * 2 * np.pi
    return np.stack([cx + r * np.cos(a), cy + r * np.sin(a)], 1)

def sample_smiley(n):
    k = np.random.multinomial(n, [0.5, 0.15, 0.15, 0.2])
    a = np.random.rand(k[0]) * 2 * np.pi
    face = 2.0 * np.stack([np.cos(a), np.sin(a)], 1)
    eye_l, eye_r = _disc(k[1], -0.7, 0.6, 0.25), _disc(k[2], 0.7, 0.6, 0.25)
    m = np.deg2rad(200 + 140 * np.random.rand(k[3]))
    mouth = 1.2 * np.stack([np.cos(m), np.sin(m)], 1)
    pts = np.concatenate([face, eye_l, eye_r, mouth]) + 0.03 * np.random.randn(n, 2)
    return _t(pts[np.random.permutation(n)])

In [ ]:
# ✔️ Checkpoint
_s = sample_smiley(4000)
assert torch.is_tensor(_s) and _s.shape == (4000, 2) and _s.dtype == torch.float32, "return a float32 tensor of shape (n, 2)"
assert torch.isfinite(_s).all() and _s.abs().max() < 3.0, "points should live inside [-3, 3]^2"
_r = _s.norm(dim=1)
assert ((_r > 1.85) & (_r < 2.15)).float().mean() > 0.4, "about half the points should be on the face circle (radius 2)"
TARGETS["smiley"] = sample_smiley
fig, ax = plt.subplots(figsize=(3, 3)); scatter(ax, _s, "smiley"); plt.show()
print("✔️ Smiley approved by Buzz. 🐝")

> **🍯 Key takeaway:** A generative model only needs *samples* of the data, never a formula for its density. In 2D you can see every success and every failure with your own eyes.

## Part 2 — The road to flow matching: VAE → DDPM → score matching → flow matching 🗺️

All four families answer one question: *how do I turn samples from an easy distribution (a Gaussian) into samples from $p_{\text{data}}$?*

**1. VAE (Kingma & Welling, 2014).** Learn an encoder $q_\phi(z\mid x)$ and decoder $p_\theta(x\mid z)$ by maximizing the ELBO
$$\log p_\theta(x) \;\ge\; \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x\mid z)] - \mathrm{KL}\big(q_\phi(z\mid x)\,\|\,\mathcal{N}(0,I)\big).$$
Generation is **one big jump** $z \mapsto x$. One network must do everything at once, and with a Gaussian likelihood the outputs tend to be blurry.

**2. Diffusion / DDPM (Sohl-Dickstein et al., 2015; Ho et al., 2020).** Fix a *forward* process that slowly adds noise,
$$q(x_t \mid x_0) = \mathcal{N}\!\big(\sqrt{\bar\alpha_t}\,x_0,\;(1-\bar\alpha_t) I\big),\qquad x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon,$$
and learn to undo it in **many small steps** with a network $\varepsilon_\theta(x_t, t)$ trained by the simple loss $\mathbb{E}\,\|\varepsilon - \varepsilon_\theta(x_t,t)\|^2$. You can view it as a deep hierarchical VAE whose encoder is fixed. (DDPM convention: $t=0$ is data, $t=T$ is noise.)

**3. Score matching (Hyvärinen, 2005; Vincent, 2011; Song & Ermon, 2019; Song et al., 2021).** The *score* $\nabla_x \log p_t(x)$ points uphill in density. Denoising score matching shows that predicting the noise *is* estimating the score: $\nabla_x \log p_t(x_t) \approx -\varepsilon_\theta(x_t,t)/\sqrt{1-\bar\alpha_t}$. Song et al. (2021) reframed diffusion as an SDE and showed there is also a deterministic **probability-flow ODE** with the same marginals — so sampling can be "just integrate an ODE".

**4. Flow matching / rectified flow (Lipman et al., 2023; Liu et al., 2023; Albergo & Vanden-Eijnden, 2023).** If the end product is an ODE $\frac{dx}{dt} = v(x,t)$ anyway, *why not learn the velocity field directly?* Pick a simple path between noise $x_0$ and data $x_1$ — e.g. a **straight line** — and regress the network onto that path's velocity. No SDEs, no noise schedules to tune, and straight paths are cheap to integrate.

| | What is learned | Training signal | Sampling |
|---|---|---|---|
| VAE | decoder $p_\theta(x\mid z)$ | ELBO | 1 network call |
| DDPM | noise $\varepsilon_\theta(x_t,t)$ | MSE on noise | ~100–1000 stochastic steps |
| Score SDE | score $s_\theta(x,t)$ | denoising score matching | reverse SDE *or* probability-flow ODE |
| Flow matching | velocity $v_\theta(x,t)$ | MSE on path velocity | ODE solve, often 10–50 steps; fewer after reflow/distillation |

⚠️ **Time convention in this lab:** for flow matching we follow Lipman et al.: **$t=0$ is noise, $t=1$ is data.** (DDPM, in Part 6, runs the other way. Always check the convention when reading papers!)

> 🐝 Buzz's one-liner: *diffusion, score matching and Gaussian-path flow matching are the same family seen through different parameterizations.* We'll prove a piece of that in Part 6.

> **🍯 Key takeaway:** A VAE jumps from noise to data in one step. Diffusion, score models and flow matching take many small steps along a learned vector field, and they differ mainly in *what the network predicts*.

## Part 3 — Conditional flow matching with straight paths ✏️

**Intuition.** Grab a noise point $x_0 \sim \mathcal{N}(0,I)$ and a data point $x_1 \sim p_{\text{data}}$, and connect them with a straight line traveled at constant speed:
$$x_t = (1-t)\,x_0 + t\,x_1, \qquad \frac{dx_t}{dt} = x_1 - x_0 .$$
Millions of such random lines cross each other. At any location $x$ and time $t$, the **marginal velocity field** is the *average* direction of all the lines passing through that point:
$$v^\star(x, t) = \mathbb{E}\big[\,x_1 - x_0 \;\big|\; x_t = x\,\big].$$
Integrating $\dot x = v^\star(x,t)$ from $t=0$ to $t=1$ transports $\mathcal{N}(0,I)$ *exactly* onto $p_{\text{data}}$ (it preserves the marginals of the interpolation — Liu et al., 2023; Lipman et al., 2023).

**The trick.** We can't compute $v^\star$ (it needs the data density), but we never have to. The **conditional flow matching** loss regresses onto the per-pair velocity:
$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t\sim U[0,1],\;x_0\sim\mathcal{N},\;x_1\sim p_{\text{data}}} \big\|\,v_\theta(x_t, t) - (x_1 - x_0)\,\big\|^2 .$$
Because the minimizer of a squared error is the conditional mean, $\mathcal{L}_{\text{CFM}} = \mathbb{E}\|v_\theta - v^\star\|^2 + \text{const}$: **same gradients as the intractable marginal loss** (Lipman et al., 2023, Thm. 2). That's the whole algorithm. 🍯

Note the loss **never reaches zero**: lines cross, so the target $x_1 - x_0$ is ambiguous at a given $(x_t,t)$. The irreducible part of the loss measures how "crossed" the paths are — remember this for reflow in Part 5!

### The velocity network

A small residual MLP with two input tricks:
- The time $t \in [0,1]$ enters through **sinusoidal features** (as in Transformers / DDPM), which let the network represent fields that change quickly in $t$:
$$\text{emb}(t) = \big[\sin(\omega_k \cdot 1000t),\ \cos(\omega_k \cdot 1000t)\big]_{k},\qquad \omega_k = 10000^{-k/K}.$$
- The position $x$ is augmented with **random Fourier features** $[\sin(Bx), \cos(Bx)]$, $B_{ij}\sim\mathcal{N}(0, 2^2)$ fixed at init (Tancik et al., 2020). Plain MLPs are biased toward smooth functions ("spectral bias"); our targets have sharp edges (checkerboard!) and thin curves (antennae!), and the velocity field near $t=1$ must be just as sharp. Curious? Train `VelocityMLP(ff=0)` and watch the samples blur.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim=64, scale=1000.0):
        super().__init__()
        half = dim // 2
        self.register_buffer("freqs", torch.exp(-math.log(10000.0) * torch.arange(half) / half))
        self.scale = scale
    def forward(self, t):                       # t: (B,) in [0, 1]
        ang = self.scale * t[:, None] * self.freqs[None, :]
        return torch.cat([ang.sin(), ang.cos()], dim=-1)

class VelocityMLP(nn.Module):
    """v_theta(x, t): R^2 x [0,1] -> R^2, a residual MLP with time injected in every block."""
    def __init__(self, dim=2, hidden=256, depth=3, t_dim=64, ff=64, ff_scale=2.0):
        super().__init__()
        self.ff = ff
        self.register_buffer("B", torch.randn(dim, ff) * ff_scale)   # fixed random Fourier frequencies
        self.temb = nn.Sequential(SinusoidalTimeEmbedding(t_dim), nn.Linear(t_dim, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
        self.inp = nn.Linear(dim + 2 * ff, hidden)
        self.blocks = nn.ModuleList([nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, hidden)) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, dim))
    def forward(self, x, t):
        if t.dim() == 0:
            t = t.expand(x.shape[0])
        te = self.temb(t)
        if self.ff:
            proj = x @ self.B
            x = torch.cat([x, proj.sin(), proj.cos()], dim=-1)
        h = self.inp(x)
        for blk in self.blocks:
            h = h + blk(h + te)
        return self.out(h)

_m = VelocityMLP()
print(f"VelocityMLP parameters: {sum(p.numel() for p in _m.parameters()):,}")
print("output shape:", _m(torch.randn(8, 2), torch.rand(8)).shape)

### 🧪 Your Turn #2 — the CFM loss

Implement the two functions below.

- `interpolate(x0, x1, t)`: the straight path $x_t = (1-t)x_0 + t x_1$. ⚠️ `t` has shape `(B,)` but `x0` has shape `(B, 2)` — reshape `t` so it broadcasts.
- `cfm_loss(model, x1, x0=None)`: if `x0` is `None`, draw it from $\mathcal{N}(0, I)$ (`torch.randn_like`). Draw `t ~ U[0,1]` with shape `(B,)`, build $x_t$, and return the mean squared error between `model(x_t, t)` and the target $x_1 - x_0$. (The optional `x0` argument lets us reuse this loss for reflow, where noise and data come **paired**.)

In [ ]:
def interpolate(x0, x1, t):
    # TODO: reshape t to (B, 1) and return the straight-line interpolation
    pass

def cfm_loss(model, x1, x0=None):
    # TODO 1: if x0 is None, sample Gaussian noise like x1
    # TODO 2: t ~ U[0, 1], shape (B,)  (use torch.rand with device=x1.device)
    # TODO 3: x_t = interpolate(x0, x1, t)
    # TODO 4: target velocity = x1 - x0
    # TODO 5: return F.mse_loss(model(x_t, t), target)
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def interpolate(x0, x1, t):
    t = t.view(-1, *([1] * (x0.dim() - 1)))      # (B,) -> (B, 1, ...) works for points AND images
    return (1 - t) * x0 + t * x1

def cfm_loss(model, x1, x0=None):
    if x0 is None:
        x0 = torch.randn_like(x1)
    t = torch.rand(x1.shape[0], device=x1.device)
    xt = interpolate(x0, x1, t)
    return F.mse_loss(model(xt, t), x1 - x0)

In [ ]:
# ✔️ Checkpoint
_x0, _x1 = torch.randn(64, 2), torch.randn(64, 2)
assert torch.allclose(interpolate(_x0, _x1, torch.zeros(64)), _x0), "t=0 should give the noise x0"
assert torch.allclose(interpolate(_x0, _x1, torch.ones(64)), _x1), "t=1 should give the data x1"
_t0, _h = torch.rand(64), 1e-3                      # finite-difference derivative == x1 - x0
_fd = (interpolate(_x0, _x1, _t0 + _h) - interpolate(_x0, _x1, _t0)) / _h
assert torch.allclose(_fd, _x1 - _x0, atol=1e-2), "d x_t / dt should equal x1 - x0"

class _Oracle(nn.Module):   # cheats: knows x0 and x1, so the loss should be exactly 0
    def forward(self, xt, t):
        return _x1 - _x0
assert cfm_loss(_Oracle(), _x1, _x0).item() < 1e-10, "with paired x0 and a perfect model, the loss must be 0"
_l = cfm_loss(VelocityMLP(), _x1)
assert _l.dim() == 0 and torch.isfinite(_l), "loss must be a finite scalar"
print("✔️ CFM loss looks right!")

### Training loop

Plain AdamW with a cosine learning-rate schedule. Each step draws a **fresh** batch from the target — in 2D we have infinite data. `pair_sampler` (used in Part 5) returns `(x0, x1)` pairs instead.

In [ ]:
FM_STEPS = 300 if FAST else 3000
BATCH = 1024

def train_fm(sampler=None, steps=FM_STEPS, batch=BATCH, lr=2e-3, model=None, pair_sampler=None, loss_fn=None, desc=""):
    seed_everything(0)
    model = model if model is not None else VelocityMLP()
    model = model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps, eta_min=lr * 0.05)
    loss_fn = loss_fn or cfm_loss
    losses, t0 = [], time.time()
    for step in range(steps):
        if pair_sampler is not None:
            x0, x1 = pair_sampler(batch)
            loss = loss_fn(model, x1.to(DEVICE), x0.to(DEVICE))
        else:
            loss = loss_fn(model, sampler(batch).to(DEVICE))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item())
    print(f"  {desc:<14s} {steps} steps in {time.time() - t0:5.1f}s | final loss (avg last 100) = {np.mean(losses[-100:]):.4f}")
    return model.eval(), losses

models, curves = {}, {}
for name in ["moons", "spiral", "checkerboard", "heart", "bee", "smiley"]:
    models[name], curves[name] = train_fm(TARGETS[name], desc=name)

plt.figure(figsize=(7, 3.5))
for name, c in curves.items():
    plt.plot(np.convolve(c, np.ones(50) / 50, mode="valid"), label=name, lw=1.5)
plt.xlabel("step"); plt.ylabel("CFM loss (smoothed)"); plt.title("The loss plateaus well above 0 — paths cross!"); plt.legend(ncol=2); plt.show()

> **🍯 Key takeaway:** Flow matching means: pick a random point on a random noise→data straight line and regress the network onto that line's velocity $x_1 - x_0$. The conditional loss has the same gradient as the intractable marginal one.

## Part 4 — ODE samplers: Euler, Heun, and the steps-vs-quality tradeoff 🧭

Sampling = solving $\frac{dx}{dt} = v_\theta(x,t)$ from $t=0$ (noise) to $t=1$ (data). With step size $h = 1/N$:

**Euler (1st order, 1 network call/step):** $\;x_{t+h} = x_t + h\,v_\theta(x_t, t)$

**Heun / explicit trapezoid (2nd order RK, 2 calls/step):**
$$\tilde x = x_t + h\,v_\theta(x_t,t),\qquad x_{t+h} = x_t + \tfrac{h}{2}\big[v_\theta(x_t,t) + v_\theta(\tilde x, t+h)\big].$$

Euler's global error is $O(h)$; Heun's is $O(h^2)$. But cost is measured in **NFE** (number of function evaluations = network calls), so Heun with $N$ steps costs as much as Euler with $2N$. Which wins depends on how *curved* the trajectories are.

### 🧪 Your Turn #3 — implement a Heun step

`euler_step` is given. Fill in `heun_step` (predictor = an Euler step, corrector = average the two slopes). Then the generic `sample_ode` loop uses whichever method you ask for.

In [ ]:
def _call(model, x, t):
    return model(x, t.expand(x.shape[0]))

def euler_step(model, x, t, dt):
    return x + dt * _call(model, x, t)

def heun_step(model, x, t, dt):
    # TODO 1: v1 = velocity at (x, t)
    # TODO 2: x_pred = Euler predictor
    # TODO 3: v2 = velocity at (x_pred, t + dt)
    # TODO 4: return x + dt * average of v1 and v2
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def heun_step(model, x, t, dt):
    v1 = _call(model, x, t)
    x_pred = x + dt * v1
    v2 = _call(model, x_pred, t + dt)
    return x + 0.5 * dt * (v1 + v2)

In [ ]:
STEPPERS = {"euler": euler_step, "heun": heun_step}

@torch.no_grad()
def sample_ode(model, x0, n_steps=50, method="euler", return_traj=False):
    """Integrate dx/dt = v(x, t) from t=0 to t=1. Returns x1 (and the trajectory, shape (n_steps+1, B, d))."""
    ts = torch.linspace(0.0, 1.0, n_steps + 1, device=x0.device)
    x, traj = x0, [x0]
    for i in range(n_steps):
        x = STEPPERS[method](model, x, ts[i], ts[i + 1] - ts[i])
        if return_traj:
            traj.append(x)
    return (x, torch.stack(traj)) if return_traj else x

NFE_PER_STEP = {"euler": 1, "heun": 2}

In [ ]:
# ✔️ Checkpoint — test the solvers on ODEs with known answers (no neural net needed!)
_decay = lambda x, t: -x                      # dx/dt = -x        ->  x(1) = x0 * e^{-1}
_drift = lambda x, t: t[:, None].expand_as(x) # dx/dt = t         ->  x(1) = x0 + 1/2
_x0 = torch.ones(5, 2)
err = {m: (sample_ode(_decay, _x0, 10, m) - _x0 * math.exp(-1)).abs().max().item() for m in STEPPERS}
print("decay ODE, 10 steps -> abs error:", {k: f"{v:.2e}" for k, v in err.items()})
assert err["euler"] < 0.03, "Euler should be roughly right (error ~0.02 with 10 steps)"
assert err["heun"] < err["euler"] / 10, "Heun (2nd order) should be >10x more accurate here"
assert torch.allclose(sample_ode(_drift, _x0, 4, "heun"), _x0 + 0.5, atol=1e-6), "Heun integrates linear-in-t fields exactly"
print("✔️ Solvers verified against closed-form solutions.")

### Watch the particles fly 🐝

Below: 400 noise particles integrated with 100 Heun steps through the learned bee field, and a frame grid of the whole population evolving over time.

In [ ]:
N_VIS = 2000 if FAST else 8000
seed_everything(1)
z = torch.randn(N_VIS, 2, device=DEVICE)
x1, traj = sample_ode(models["bee"], z, n_steps=100, method="heun", return_traj=True)
traj = traj.cpu()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ax = axes[0]
for i in range(400):
    ax.plot(traj[:, i, 0], traj[:, i, 1], lw=0.5, alpha=0.35, c=HIVE[4])
ax.scatter(traj[0, :400, 0], traj[0, :400, 1], s=4, c="gray", label="t=0 (noise)", zorder=3)
ax.scatter(traj[-1, :400, 0], traj[-1, :400, 1], s=4, c=HIVE[0], label="t=1 (sample)", zorder=3)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.legend(loc="lower left"); ax.set_title("Trajectories: noise → bee")
scatter(axes[1], x1, "Final samples (100 Heun steps)", s=1.2)
plt.tight_layout(); plt.show()

frames = [0, 10, 20, 30, 45, 60, 75, 90, 100]
fig, axes = plt.subplots(1, len(frames), figsize=(2.2 * len(frames), 2.4))
for ax, f in zip(axes, frames):
    scatter(ax, traj[f], f"t = {f / 100:.2f}", s=0.6, c=plt.cm.viridis(f / 100))
plt.suptitle("Frame grid: the distribution flowing from Gaussian to Buzz", y=1.05); plt.show()

### 🎬 The wow moment: Buzz assembles himself

Press ▶ below. Each frame is the *whole population* at one moment of the ODE. The color shifts from purple (noise) to yellow (data).

In [ ]:
from matplotlib import animation
from IPython.display import HTML
plt.rcParams["animation.embed_limit"] = 30          # MB; this animation is only a few MB

sub = traj[::4, :3000]                              # 26 frames x 3000 particles
frame_ids = list(range(len(sub))) + [len(sub) - 1] * 4   # linger on the final bee
fig, ax = plt.subplots(figsize=(4.6, 4.6), dpi=80)
sc = ax.scatter(sub[0, :, 0], sub[0, :, 1], s=2.5, color=plt.cm.viridis(0.0), alpha=0.7, linewidths=0)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
ttl = ax.set_title("t = 0.00")

def update(k):
    i = frame_ids[k]
    frac = i / (len(sub) - 1)
    sc.set_offsets(sub[i].numpy()); sc.set_color(plt.cm.viridis(frac)); ttl.set_text(f"t = {frac:.2f}")
    return sc, ttl

anim = animation.FuncAnimation(fig, update, frames=len(frame_ids), interval=110, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

Notice the trajectories are **curved**, even though every training path was a straight line! Each particle follows the *average* of many crossing straight lines, and averages of lines bend. Hold that thought.

### A yardstick: sliced Wasserstein distance

To compare samplers quantitatively we need a distance between two point clouds. The **sliced Wasserstein-2 distance** projects both clouds onto random directions $\theta$, where 1D optimal transport is just *sorting*:
$$\mathrm{SW}_2(P,Q)^2 = \mathbb{E}_{\theta \sim \text{Unif}(S^{1})}\, W_2^2\big(\theta^\top_\# P,\ \theta^\top_\# Q\big).$$
Lower is better; two independent sample sets from the same distribution give a small but nonzero "noise floor".

In [ ]:
def sliced_wasserstein(a, b, n_proj=256, seed=0):
    a, b = a.detach().cpu().float(), b.detach().cpu().float()
    g = torch.Generator().manual_seed(seed)
    theta = torch.randn(n_proj, a.shape[1], generator=g)
    theta = theta / theta.norm(dim=1, keepdim=True)
    pa, pb = (a @ theta.T).sort(0).values, (b @ theta.T).sort(0).values
    return ((pa - pb) ** 2).mean().sqrt().item()

N_EVAL = 2000 if FAST else 5000
REF = {name: TARGETS[name](N_EVAL) for name in models}
floor = {name: sliced_wasserstein(TARGETS[name](N_EVAL), REF[name]) for name in models}
print("noise floor (two fresh data samples):", {k: round(v, 3) for k, v in floor.items()})

### 🔮 Predict-then-run

Before running the next cell, write down your guess: **with a single Euler step, what will the samples look like?** A blurry bee? Pure noise? Something else? *(Hint: what is $v^\star(x, 0)$ when $x_0$ and $x_1$ are independent?)*

In [ ]:
STEP_LIST = [1, 2, 4, 8, 16, 64]
names = list(models)
fig, axes = plt.subplots(len(names), len(STEP_LIST), figsize=(2.1 * len(STEP_LIST), 2.1 * len(names)))
seed_everything(2)
z = torch.randn(N_EVAL, 2, device=DEVICE)
for r, name in enumerate(names):
    for c, n in enumerate(STEP_LIST):
        xs = sample_ode(models[name], z, n_steps=n, method="euler")
        scatter(axes[r, c], xs, f"{n} Euler step{'s' if n > 1 else ''}" if r == 0 else None, s=0.5)
    axes[r, 0].set_ylabel(name)
plt.suptitle("More steps → sharper samples", y=1.01); plt.tight_layout(); plt.show()

**Reveal.** With one Euler step, $x_1 = x_0 + v_\theta(x_0, 0)$. At $t=0$ the point $x_t = x_0$ carries **no information** about which data point it's paired with, so the optimal velocity is $v^\star(x,0) = \mathbb{E}[x_1] - x$. Therefore $x_0 + v^\star(x_0,0) = \mathbb{E}[x_1]$: **every sample collapses onto the data mean!** (Look at the first column: a tight blob near the center of mass.) Few-step sampling fails exactly because the flow is curved.

Now let's quantify quality vs. compute for Euler and Heun on the same NFE budget.

In [ ]:
NFE_LIST = [2, 4, 8, 16, 32, 64]
results = {}
for name in ["moons", "checkerboard", "bee"]:
    for method in STEPPERS:
        sw = []
        for nfe in NFE_LIST:
            xs = sample_ode(models[name], z, n_steps=max(1, nfe // NFE_PER_STEP[method]), method=method)
            sw.append(sliced_wasserstein(xs, REF[name]))
        results[(name, method)] = sw

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=False)
for ax, name in zip(axes, ["moons", "checkerboard", "bee"]):
    for method, col in zip(STEPPERS, [HIVE[2], HIVE[4]]):
        ax.plot(NFE_LIST, results[(name, method)], "o-", c=col, label=method)
    ax.axhline(floor[name], ls="--", c="gray", lw=1, label="noise floor")
    ax.set_xscale("log", base=2); ax.set_xlabel("NFE (network calls)"); ax.set_title(name)
axes[0].set_ylabel("sliced W2 to data ↓"); axes[0].legend(); plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** Sampling is solving an ODE. Count cost in network calls (NFE). Higher-order solvers pay off when the flow is curved, and on a curved flow a single step lands on the data mean.

## Part 5 — Reflow: straightening the flow 📏

**Why are trajectories curved?** Independent pairing $(x_0, x_1) \sim \mathcal{N} \times p_{\text{data}}$ creates crossing lines, and the learned field averages them. **Rectified flow** (Liu et al., 2023) fixes the *coupling*:

1. Train flow #1 ("1-rectified flow") as above.
2. Simulate it: draw $x_0 \sim \mathcal{N}$ and compute $\hat x_1 = \mathrm{ODE}_1(x_0)$ accurately (many steps). Because an ODE flow is a deterministic map, distinct $x_0$'s are sent to distinct $\hat x_1$'s along non-crossing paths.
3. Train flow #2 ("2-rectified flow") with the **same CFM loss** but on the *paired* data $(x_0, \hat x_1)$.

Liu et al. prove that rectification never increases convex transport costs, and that the **straightness**
$$S(v) = \int_0^1 \mathbb{E}\,\big\|(x_1 - x_0) - v(x_t, t)\big\|^2\,dt \quad\text{(along the ODE trajectories)}$$
is driven toward zero by repeated reflow. If $S=0$ the paths are perfectly straight and **a single Euler step is exact**.

Cost: reflow needs one extra round of simulation + training, and errors in flow #1 get baked into flow #2 (its target is flow #1's samples, not the true data). That's the price of few-step speed.

### 🧪 Your Turn #4 — measure straightness

Implement `straightness(model, n, n_steps)`:
1. draw `x0 ~ N(0, I)` (shape `(n, 2)`, on `DEVICE`),
2. integrate with **Euler** for `n_steps`, recording the velocity `v` at each step,
3. with `x1` the final point, return the mean over steps and particles of $\|(x_1 - x_0) - v\|^2$.

In [ ]:
@torch.no_grad()
def straightness(model, n=2000, n_steps=64):
    # TODO 1: x0 = torch.randn(n, 2, device=DEVICE); ts = torch.linspace(0, 1, n_steps + 1, device=DEVICE)
    # TODO 2: loop: v = _call(model, x, ts[i]); store v; x = x + (ts[i+1] - ts[i]) * v
    # TODO 3: x1 = final x; return mean of ||(x1 - x0) - v_i||^2 over all steps i and particles (as a float)
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
@torch.no_grad()
def straightness(model, n=2000, n_steps=64):
    x0 = torch.randn(n, 2, device=DEVICE)
    ts = torch.linspace(0, 1, n_steps + 1, device=DEVICE)
    x, vs = x0, []
    for i in range(n_steps):
        v = _call(model, x, ts[i]); vs.append(v)
        x = x + (ts[i + 1] - ts[i]) * v
    vs = torch.stack(vs)                                  # (n_steps, n, 2)
    return ((x - x0)[None] - vs).pow(2).sum(-1).mean().item()

In [ ]:
# ✔️ Checkpoint — a constant field is perfectly straight; a rotating field is not
class _Const(nn.Module):
    def forward(self, x, t): return torch.ones_like(x)
class _Rotate(nn.Module):
    def forward(self, x, t): return torch.stack([-x[:, 1], x[:, 0]], 1) * 3
assert straightness(_Const(), 500) < 1e-10, "a constant velocity field has straightness 0"
assert straightness(_Rotate(), 500) > 1.0, "a swirling field should be far from straight"
print(f"✔️ straightness: const = {straightness(_Const(), 500):.1e}, swirl = {straightness(_Rotate(), 500):.2f}")

### Run reflow

We'll rectify the field for `REFLOW_TARGET` (try `"bee"` or `"checkerboard"` too!). We initialize flow #2 from flow #1's weights — a common trick that makes reflow converge fast.

🔮 **Predict-then-run:** after one round of reflow, will **1-step** sampling produce a sharp distribution? Will the 64-step samples get *better* or slightly *worse* than flow #1's?

In [ ]:
REFLOW_TARGET = "moons"
N_PAIRS = 5000 if FAST else 30000
REFLOW_STEPS = 300 if FAST else 2500

base = models[REFLOW_TARGET]
seed_everything(3)
pair_x0 = torch.randn(N_PAIRS, 2, device=DEVICE)
t0 = time.time()
pair_x1 = torch.cat([sample_ode(base, chunk, n_steps=50, method="heun") for chunk in pair_x0.split(10000)])
print(f"simulated {N_PAIRS} (noise, sample) pairs in {time.time() - t0:.1f}s")

def pair_sampler(b):
    idx = torch.randint(0, N_PAIRS, (b,), device=DEVICE)
    return pair_x0[idx], pair_x1[idx]

reflow, reflow_curve = train_fm(model=copy.deepcopy(base), pair_sampler=pair_sampler, steps=REFLOW_STEPS, lr=5e-4, desc="2-rectified")
print(f"straightness  1-rectified: {straightness(base):.4f}   |   2-rectified: {straightness(reflow):.4f}")
print(f"final CFM loss 1-rect: {np.mean(curves[REFLOW_TARGET][-100:]):.4f}  vs  2-rect (paired): {np.mean(reflow_curve[-100:]):.4f}  ← paired lines barely cross")

In [ ]:
seed_everything(4)
zv = torch.randn(300, 2, device=DEVICE)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
for ax, (m, title) in zip(axes, [(base, "1-rectified flow"), (reflow, "2-rectified flow (after reflow)")]):
    _, tr = sample_ode(m, zv, n_steps=64, method="heun", return_traj=True); tr = tr.cpu()
    for i in range(300):
        ax.plot(tr[:, i, 0], tr[:, i, 1], lw=0.6, alpha=0.5, c=HIVE[4])
    ax.scatter(tr[-1, :, 0], tr[-1, :, 1], s=5, c=HIVE[0], zorder=3)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.set_title(f"{title}\nstraightness = {straightness(m):.4f}")
plt.tight_layout(); plt.show()

few = [1, 2, 4, 64]
fig, axes = plt.subplots(2, len(few), figsize=(2.6 * len(few), 5.4))
for r, (m, lab) in enumerate([(base, "1-rect"), (reflow, "2-rect")]):
    for c, n in enumerate(few):
        xs = sample_ode(m, z, n_steps=n, method="euler")
        scatter(axes[r, c], xs, f"{lab}: {n} step{'s' if n > 1 else ''}\nSW={sliced_wasserstein(xs, REF[REFLOW_TARGET]):.3f}", s=0.5)
plt.tight_layout(); plt.show()

**What you should see:** reflow trajectories are visibly straighter, and the 1–4 step samples of the 2-rectified flow are dramatically better (1 step is no longer a collapsed blob). With many steps, flow #2 may be slightly *worse* than flow #1 — it learned to imitate flow #1's samples, errors included. Production systems combine reflow with **distillation** (e.g. InstaFlow, Liu et al., 2024) to get high-quality 1-step generators.

> **🍯 Key takeaway:** Reflow pairs each noise sample with the model's own output, so paths stop crossing and straighten out. One extra round of training buys few-step (even 1-step) sampling.

## Part 6 — DDPM on the same data (and why it's the same family) 🔁

Now the classic. DDPM uses discrete time $t \in \{0,\dots,T-1\}$ with **$t=0$ = data**, a variance schedule $\beta_t$, $\alpha_t = 1-\beta_t$, $\bar\alpha_t = \prod_{s\le t}\alpha_s$:

**Training (Ho et al., 2020, "$L_\text{simple}$"):** $\quad x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon, \qquad \mathcal{L} = \mathbb{E}_{t,x_0,\varepsilon}\|\varepsilon - \varepsilon_\theta(x_t,t)\|^2$

**Ancestral sampling (stochastic, $T$ steps):**
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t,t)\Big) + \sqrt{\beta_t}\,z,\quad z\sim\mathcal{N}(0,I)$$

**DDIM (Song, Meng & Ermon, 2021, deterministic, any subsequence of steps):** predict $\hat x_0 = (x_t - \sqrt{1-\bar\alpha_t}\,\varepsilon_\theta)/\sqrt{\bar\alpha_t}$, then jump to $x_{t'} = \sqrt{\bar\alpha_{t'}}\,\hat x_0 + \sqrt{1-\bar\alpha_{t'}}\,\varepsilon_\theta$. DDIM is a discretization of the probability-flow ODE — an ODE sampler, just like ours.

We reuse the same `VelocityMLP` architecture (it's just "a network of $(x, t)$"), feeding $t/T$ as the time input.

In [ ]:
T_DDPM = 200
betas = torch.linspace(1e-4, 0.05, T_DDPM, device=DEVICE)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)
print(f"sqrt(abar_T) = {abar[-1].sqrt().item():.3f}   (signal left at t=T; ~0 means pure noise)")

def ddpm_loss(model, x_data, x0_unused=None):
    t = torch.randint(0, T_DDPM, (x_data.shape[0],), device=x_data.device)
    eps = torch.randn_like(x_data)
    a = abar[t][:, None]
    xt = a.sqrt() * x_data + (1 - a).sqrt() * eps
    return F.mse_loss(model(xt, t.float() / T_DDPM), eps)

@torch.no_grad()
def ddpm_sample(model, n, seed=0):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.randn(n, 2, device=DEVICE, generator=g)
    for t in reversed(range(T_DDPM)):
        eps = model(x, torch.full((n,), t / T_DDPM, device=DEVICE))
        x = (x - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn(x.shape, device=DEVICE, generator=g)
    return x

@torch.no_grad()
def ddim_sample(model, n, n_steps=20, seed=0):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.randn(n, 2, device=DEVICE, generator=g)
    ts = torch.linspace(T_DDPM - 1, 0, n_steps).round().long().tolist()
    for i, t in enumerate(ts):
        eps = model(x, torch.full((n,), t / T_DDPM, device=DEVICE))
        x0_hat = (x - (1 - abar[t]).sqrt() * eps) / abar[t].sqrt()
        a_next = abar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=DEVICE)
        x = a_next.sqrt() * x0_hat + (1 - a_next).sqrt() * eps
    return x

ddpm_model, ddpm_curve = train_fm(TARGETS[REFLOW_TARGET], loss_fn=ddpm_loss, desc="DDPM eps-pred")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
panels = [("data", REF[REFLOW_TARGET]),
          (f"DDPM ancestral\n{T_DDPM} NFE", ddpm_sample(ddpm_model, N_EVAL)),
          ("DDIM\n10 NFE", ddim_sample(ddpm_model, N_EVAL, 10)),
          ("Flow matching Euler\n10 NFE", sample_ode(base, z, 10)),
          ("Reflow Euler\n2 NFE", sample_ode(reflow, z, 2))]
for ax, (title, xs) in zip(axes, panels):
    sw = "" if title == "data" else f"\nSW={sliced_wasserstein(xs, REF[REFLOW_TARGET]):.3f}"
    scatter(ax, xs, title + sw, s=0.5)
plt.tight_layout(); plt.show()

nfes = [1, 2, 4, 8, 16, 32, 64]
curves_nfe = {
    "DDIM (eps-pred)": [sliced_wasserstein(ddim_sample(ddpm_model, N_EVAL, k), REF[REFLOW_TARGET]) for k in nfes],
    "FM Euler": [sliced_wasserstein(sample_ode(base, z, k), REF[REFLOW_TARGET]) for k in nfes],
    "Reflow Euler": [sliced_wasserstein(sample_ode(reflow, z, k), REF[REFLOW_TARGET]) for k in nfes],
}
plt.figure(figsize=(6.5, 4))
for (lab, ys), col in zip(curves_nfe.items(), [HIVE[5], HIVE[2], HIVE[3]]):
    plt.plot(nfes, ys, "o-", label=lab, c=col)
plt.axhline(sliced_wasserstein(ddpm_sample(ddpm_model, N_EVAL), REF[REFLOW_TARGET]), c=HIVE[1], ls=":", label=f"DDPM ancestral ({T_DDPM} NFE)")
plt.axhline(floor[REFLOW_TARGET], c="gray", ls="--", lw=1, label="noise floor")
plt.xscale("log", base=2); plt.xlabel("NFE"); plt.ylabel("sliced W2 ↓"); plt.title(f"Quality vs compute on '{REFLOW_TARGET}'"); plt.legend(fontsize=9); plt.show()

**Why does DDIM plateau above ancestral DDPM here?** Two effects compound. (1) ε-prediction is badly conditioned at high noise: the implied clean point is $\hat x_0 = (x_t - \sqrt{1-\bar\alpha_t}\,\hat\varepsilon)/\sqrt{\bar\alpha_t}$, so a small ε-error gets multiplied by $1/\sqrt{\bar\alpha_t}$ (≈13× at our $t=T$). (2) A deterministic ODE sampler carries those early errors all the way to $t=0$, whereas ancestral sampling keeps injecting fresh noise, which lets later (more accurate) steps correct earlier mistakes — Karras et al. (2022) analyze this ODE-vs-SDE tradeoff. Velocity targets (flow matching, and the closely related *v-prediction* of Salimans & Ho, 2022) stay well-scaled at every $t$, one practical reason modern systems adopted them. Training the ε-model longer shrinks the gap — try it!

### Same family, different hats 🎩

For the Gaussian straight path $x_t = (1-t)x_0 + t x_1$ (noise $x_0$, data $x_1$), knowing the velocity is the same as knowing the noise or the clean data. From $x_t$ and $v = x_1 - x_0$:
$$\hat x_0^{\text{(noise)}} = x_t - t\,v, \qquad \hat x_1^{\text{(data)}} = x_t + (1-t)\,v .$$
And since $x_t \mid x_1 \sim \mathcal{N}(t x_1, (1-t)^2 I)$, the score is
$$\nabla_x \log p_t(x_t) = -\frac{\mathbb{E}[x_0\mid x_t]}{1-t} = -\frac{x_t - t\,v^\star(x_t,t)}{1-t}.$$
So an ε-predictor, a score model, an $x$-predictor and a velocity model are **affine re-parameterizations of each other** (they differ in how the loss weights different noise levels, which matters in practice). Stable Diffusion 3 (Esser et al., 2024) is exactly a rectified-flow model trained this way at scale.

### 🧪 Your Turn #5 — convert velocity ↔ noise ↔ data ↔ score

In [ ]:
def velocity_to_noise(v, xt, t):
    # TODO: return the implied noise x0 = x_t - t v   (t has shape (B,))
    pass

def velocity_to_data(v, xt, t):
    # TODO: return the implied clean sample x1 = x_t + (1 - t) v
    pass

def velocity_to_score(v, xt, t):
    # TODO: return -(x_t - t v) / (1 - t)
    pass

In [ ]:
# ✅ Solution (peek only after trying!)
def velocity_to_noise(v, xt, t):
    return xt - t[:, None] * v

def velocity_to_data(v, xt, t):
    return xt + (1 - t[:, None]) * v

def velocity_to_score(v, xt, t):
    return -(xt - t[:, None] * v) / (1 - t[:, None])

In [ ]:
# ✔️ Checkpoint — exact recovery on a single pair, and the score of a known Gaussian
_x0, _x1, _tt = torch.randn(100, 2), torch.randn(100, 2), torch.rand(100) * 0.9
_xt, _v = interpolate(_x0, _x1, _tt), _x1 - _x0
assert torch.allclose(velocity_to_noise(_v, _xt, _tt), _x0, atol=1e-5)
assert torch.allclose(velocity_to_data(_v, _xt, _tt), _x1, atol=1e-5)
# If the data is a single point x1 = c, then p_t = N(t c, (1-t)^2 I), whose score is -(x - t c)/(1-t)^2.
_c = torch.tensor([[1.0, -2.0]]).expand(100, 2)
_xt = interpolate(_x0, _c, _tt)
_vstar = (_c - _xt) / (1 - _tt[:, None])           # true marginal velocity for a point-mass target
assert torch.allclose(velocity_to_score(_vstar, _xt, _tt), -(_xt - _tt[:, None] * _c) / (1 - _tt[:, None]) ** 2, atol=1e-4)
print("✔️ You can now translate between diffusion dialects. 🐝")

> **🍯 Key takeaway:** ε-prediction, score, clean-data prediction and velocity are interchangeable views of one Gaussian-path model, but they are not equally well-conditioned, and deterministic samplers never correct earlier mistakes.

## 🏆 Boss Level

1. **Minibatch optimal-transport flow matching.** Instead of random pairing, match each noise sample to a data sample *within the minibatch* by solving an assignment problem (`scipy.optimize.linear_sum_assignment` on the squared-distance matrix) before computing the CFM loss (Tong et al., 2024; Pooladian et al., 2023). Compare straightness and few-step SW against both the plain and the reflowed model — you get straighter paths *without* a second training round.
2. **One model, many shapes (a preview of Lab 4.2).** Make `VelocityMLP` class-conditional: add an `nn.Embedding(len(TARGETS) + 1, hidden)` for the target id, randomly drop the label 10% of the time to a "null" id, and at sampling time use $v = v_\varnothing + w\,(v_c - v_\varnothing)$. What happens to the bee as you crank $w$ from 0 to 5?
3. **Exact likelihoods.** A flow ODE gives exact log-densities via the instantaneous change of variables $\frac{d}{dt}\log p_t(x_t) = -\nabla\cdot v(x_t,t)$ (Chen et al., 2018). In 2D the divergence is cheap — compute it with two autograd calls — and render a heat map of $\log p_1(x)$ on a grid by integrating *backwards* from data to noise.

## 🍯 Key Takeaways

- **Flow matching in one line:** sample $t$, noise $x_0$ and data $x_1$; regress $v_\theta\big((1-t)x_0 + t x_1,\, t\big)$ onto $x_1 - x_0$. That really is the whole training algorithm.
- **Conditional = marginal (in expectation):** per-pair targets give the same gradients as the true marginal velocity field. The loss stays above zero because lines cross.
- **Sampling is ODE integration.** Measure cost in NFE. Euler is first order and Heun is second order. One step on a curved flow returns the data mean.
- **Reflow straightens paths** by pairing noise with the model's own samples, which makes 1–4 step sampling viable for one extra training round.
- **DDPM, score matching and Gaussian flow matching are one family** in different parameterizations. DDIM is an ODE sampler, and ancestral sampling's injected noise can correct model errors.
- **Parameterization matters in practice:** ε-prediction is ill-conditioned at high noise, while velocity targets stay well-scaled. This is one reason SD3-style models use rectified flow.
- **Everything scales:** swap the MLP for a UNet or transformer and the 2D points for image latents, and you have a modern image generator (Lab 4.2).

## 🎤 Explain it to a friend

Your friend has never heard of diffusion models. In one paragraph, with no equations, explain how a computer can turn a cloud of random dots into a drawing of a bee by repeatedly "following the arrows". Then explain why taking *one* giant step gives you a smudge in the middle instead of a bee, and what "reflow" does to fix it. (Hint: think of a crowded room where everyone walks in straight lines and the arrows show the *average* direction people move through each spot.)

## 💼 Interview corner

**Q1. In conditional flow matching we regress onto $x_1 - x_0$, a target that depends on the specific pair. Why does that recover the *marginal* velocity field, and why doesn't the training loss go to zero?**
<details><summary>Answer</summary>

The squared error is minimized by the conditional expectation, so the optimum is $v^\star(x,t) = \mathbb{E}[x_1 - x_0 \mid x_t = x]$, the marginal velocity. Equivalently, $\mathcal{L}_\text{CFM} = \mathbb{E}\|v_\theta - v^\star\|^2 + C$, where $C$ does not depend on $\theta$, so the gradients match the intractable marginal loss (Lipman et al., 2023). The constant $C$ is the variance of $x_1 - x_0$ given $x_t$. It is non-zero because many straight lines pass through the same $(x,t)$, so the loss plateaus above zero.
</details>

**Q2. Compare DDPM ancestral sampling with DDIM / probability-flow ODE sampling. When would you prefer each?**
<details><summary>Answer</summary>

Ancestral sampling is stochastic: each step denoises and then injects fresh Gaussian noise. It needs many steps but tends to correct accumulated model errors. DDIM ($\eta=0$) is a deterministic discretization of the probability-flow ODE. It allows large step skips and gives a fixed noise→sample mapping, which is useful for interpolation, inversion/editing and distillation, but it carries early errors to the end. Use ODE samplers for speed and determinism. Use stochastic (or mixed, e.g. Karras et al. 2022) samplers when you have the step budget and want maximum quality.
</details>

**Q3. Your product needs images in under 4 network evaluations. What are your options, and what are their tradeoffs?**
<details><summary>Answer</summary>

(1) Better ODE solvers (DPM-Solver, Heun, etc.): no retraining, but quality drops sharply below ~10 steps. (2) Straighten the flow (reflow, minibatch-OT coupling): needs extra training and inherits the teacher's errors. (3) Distillation: progressive distillation, consistency models / consistency distillation, or adversarial distillation (ADD) can reach 1–4 steps, at the cost of extra training, some diversity, and often a teacher model. Also distill away classifier-free guidance, which halves the per-step cost.
</details>

## 🐝 Fun fact

The first diffusion-model paper (Sohl-Dickstein et al., 2015) was inspired by **non-equilibrium thermodynamics**: slowly destroy structure with noise, then learn to reverse the process. It drew relatively little attention until Ho et al. (2020) revived the idea five years later with a simple noise-prediction loss and impressive image samples. Since then, generative modeling has come to be dominated by diffusion and flow models.

## 📚 Further reading

- Lipman, Chen, Ben-Hamu, Nickel & Le (2023). *Flow Matching for Generative Modeling.* ICLR.
- Liu, Gong & Liu (2023). *Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow.* ICLR.
- Albergo & Vanden-Eijnden (2023). *Building Normalizing Flows with Stochastic Interpolants.* ICLR.
- Lipman et al. (2024). *Flow Matching Guide and Code.* arXiv preprint.
- Ho, Jain & Abbeel (2020). *Denoising Diffusion Probabilistic Models.* NeurIPS.
- Sohl-Dickstein, Weiss, Maheswaranathan & Ganguli (2015). *Deep Unsupervised Learning using Nonequilibrium Thermodynamics.* ICML.
- Song & Ermon (2019). *Generative Modeling by Estimating Gradients of the Data Distribution.* NeurIPS.
- Song, Sohl-Dickstein, Kingma, Kumar, Ermon & Poole (2021). *Score-Based Generative Modeling through Stochastic Differential Equations.* ICLR.
- Song, Meng & Ermon (2021). *Denoising Diffusion Implicit Models.* ICLR.
- Vincent (2011). *A Connection Between Score Matching and Denoising Autoencoders.* Neural Computation.
- Kingma & Welling (2014). *Auto-Encoding Variational Bayes.* ICLR.
- Tong et al. (2024). *Improving and Generalizing Flow-Based Generative Models with Minibatch Optimal Transport.* TMLR.
- Esser et al. (2024). *Scaling Rectified Flow Transformers for High-Resolution Image Synthesis.* ICML.
- Tancik et al. (2020). *Fourier Features Let Networks Learn High Frequency Functions in Low Dimensional Domains.* NeurIPS.
- Karras, Aittala, Aila & Laine (2022). *Elucidating the Design Space of Diffusion-Based Generative Models.* NeurIPS.
- Salimans & Ho (2022). *Progressive Distillation for Fast Sampling of Diffusion Models.* ICLR.
- Chen, Rubanova, Bettencourt & Duvenaud (2018). *Neural Ordinary Differential Equations.* NeurIPS.

🐝 *Buzz says: you just built the engine inside modern image and video generators. Next stop: pixels!*